# SVD and PCA

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.06 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/07_SVD_and_PCA.ipynb)

---

## 🎯 Learning Objective

Derive the Singular Value Decomposition as the generalization of eigendecomposition to any matrix, and use it to perform Principal Component Analysis for dimensionality reduction.

---

## 📐 Theory

Eigendecomposition requires a square matrix. The **Singular Value Decomposition (SVD)**
generalizes the idea to *any* matrix $A$ (shape $m\times n$):

$$A = U\Sigma V^\top$$

- $U$ ($m\times m$) and $V$ ($n\times n$) are orthogonal (their columns are orthonormal).
- $\Sigma$ ($m\times n$) is diagonal with non-negative entries $\sigma_1 \ge \sigma_2 \ge
  \cdots \ge 0$, the **singular values**.

Geometrically: *any* linear map is "rotate ($V^\top$), scale along axes ($\Sigma$), rotate
again ($U$)." This decomposes even non-square, rank-deficient, or asymmetric matrices — the
generalization eigendecomposition couldn't offer.

### Connection to eigendecomposition

The singular values of $A$ are the square roots of the eigenvalues of $A^\top A$ (which is
always symmetric positive-semi-definite, so this is always well-defined even when $A$ itself
isn't square or symmetric).

### Truncated SVD and the Eckart-Young theorem

Keeping only the top $k$ singular values/vectors gives $A_k = U_k \Sigma_k V_k^\top$, the
*best possible* rank-$k$ approximation of $A$ in a precise sense (proved by the Eckart-Young
theorem, revisited in [Notebook 08.05](../08_Advanced_Linear_Algebra/05_low_rank_approximations.ipynb)).
The fraction of "energy" (squared Frobenius norm) captured by the top $k$ components is
$\sum_{i\le k}\sigma_i^2 \big/ \sum_i \sigma_i^2$ — this is exactly the "variance explained" number
PCA reports.

### PCA via SVD

Running SVD directly on the *centered data matrix* $X$ (rather than eigendecomposing the
covariance matrix, as in [Notebook 01.06](06_eigenvalues_eigenvectors.ipynb)) is the
numerically preferred way to compute PCA in practice, since it avoids ever forming
$X^\top X$ explicitly (which can lose precision).

---

## 👁️ Visual Intuition

Reconstructing an image from a truncated SVD, rank by rank, makes "low-rank approximation"
visceral: a handful of singular values already captures most of the recognizable structure.

In [ ]:
# Truncated SVD image compression: how few singular values are needed to recognize the image?
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Build a simple synthetic "image": concentric rings (has real low-rank structure to exploit)
size = 80
yy, xx = np.mgrid[0:size, 0:size]
center = size / 2
radius = np.sqrt((xx - center) ** 2 + (yy - center) ** 2)
image = np.sin(radius / 3.0) * np.exp(-radius / 60.0)

U, S, Vt = np.linalg.svd(image, full_matrices=False)

ranks = [1, 5, 20, size]
fig, axes = plt.subplots(1, len(ranks), figsize=(14, 4))
for ax, k in zip(axes, ranks):
    approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    energy_captured = (S[:k] ** 2).sum() / (S ** 2).sum()
    ax.imshow(approx, cmap="viridis")
    ax.set_title(f"rank {k}\n{energy_captured*100:.1f}% energy")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We compute the SVD via eigendecomposition of $A^\top A$ (the textbook definition), confirm it
matches `np.linalg.svd`, then run PCA via SVD directly on centered data.

In [ ]:
# SVD from the eigendecomposition of A^T A, then PCA via SVD on centered data
import numpy as np

def svd_from_scratch(A):
    # Singular values/right-singular-vectors come from eigendecomposing the symmetric A^T A
    AtA = A.T @ A
    eigvals, V = np.linalg.eigh(AtA)
    order = np.argsort(eigvals)[::-1]
    eigvals, V = eigvals[order], V[:, order]
    eigvals = np.clip(eigvals, 0, None)         # guard against tiny negative numerical noise
    singular_values = np.sqrt(eigvals)

    # Left-singular vectors: U_i = A v_i / sigma_i  (for nonzero sigma_i)
    U = np.zeros((A.shape[0], len(singular_values)))
    for i, sv in enumerate(singular_values):
        if sv > 1e-10:
            U[:, i] = (A @ V[:, i]) / sv
    return U, singular_values, V.T

A = np.array([[3.0, 1.0], [1.0, 3.0], [1.0, 1.0]])
U_scratch, S_scratch, Vt_scratch = svd_from_scratch(A)
U_np, S_np, Vt_np = np.linalg.svd(A, full_matrices=False)

print("Singular values (scratch):", np.round(S_scratch, 4))
print("Singular values (numpy)  :", np.round(S_np, 4))
reconstructed = U_scratch @ np.diag(S_scratch) @ Vt_scratch
print("Reconstruction A ~ U S V^T matches original A:", np.allclose(reconstructed, A))

print("\n--- PCA via SVD directly on centered data ---")
rng = np.random.default_rng(3)
X = rng.multivariate_normal([0, 0], [[3.0, 1.5], [1.5, 1.0]], size=500)
X_centered = X - X.mean(axis=0)

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
explained_variance = (S ** 2) / (X.shape[0] - 1)
print("Explained variance via SVD:", np.round(explained_variance, 3))
print("(matches the eigenvalues of the covariance matrix computed in Notebook 01.06)")

## 🤖 Why This Matters for AI

**LoRA (Low-Rank Adaptation)** fine-tunes large language models by freezing the pretrained
weight matrix $W_0$ and learning only a low-rank *update*:

$$W = W_0 + BA, \qquad B \in \mathbb{R}^{d\times r},\ A \in \mathbb{R}^{r\times d},\ r \ll d$$

This is the SVD's low-rank approximation idea turned into a training strategy: instead of
updating all $d^2$ parameters of $W$, you only learn $2dr$ parameters — often a >100x
reduction. We simulate this below: approximate a "full fine-tuning" weight update with its
best rank-$r$ approximation and measure how much of the update's effect survives.

In [ ]:
# Simulating LoRA: approximating a full weight update with a low-rank factorization
import numpy as np

rng = np.random.default_rng(0)
d = 64  # a stand-in for a hidden dimension (real LLMs: thousands)

W0 = rng.standard_normal((d, d)) * 0.02              # frozen pretrained weights
delta_W_full = rng.standard_normal((d, d)) * 0.01     # what "full fine-tuning" would have learned

U, S, Vt = np.linalg.svd(delta_W_full)

for r in [1, 4, 16, 64]:
    B = U[:, :r] * S[:r]           # (d, r)
    A_lora = Vt[:r, :]              # (r, d)
    delta_W_lora = B @ A_lora        # rank-r approximation of the full update

    full_params = d * d
    lora_params = d * r + r * d
    energy_captured = (S[:r] ** 2).sum() / (S ** 2).sum()
    approx_error = np.linalg.norm(delta_W_full - delta_W_lora) / np.linalg.norm(delta_W_full)

    print(f"rank r={r:3d}: {lora_params:6d}/{full_params} params "
          f"({100*lora_params/full_params:5.1f}%), "
          f"energy captured={energy_captured*100:5.1f}%, "
          f"relative approx error={approx_error:.3f}")

## 🏋️ Exercises

### Warm-up
1. Compute the SVD of $\begin{pmatrix}3&0\\0&-2\end{pmatrix}$ by hand (note: singular values are always non-negative, even though eigenvalues here are negative). Verify with `np.linalg.svd`.

### Practice
2. For a random $5\times3$ matrix, verify `svd_from_scratch` matches `np.linalg.svd` (up to sign flips in the columns), and confirm $\sum_i \sigma_i^2$ equals the squared Frobenius norm of the matrix.

### Challenge
3. Using the ring image from the Visual Intuition cell, plot 'energy captured' (cumulative $\sum_{i\le k}\sigma_i^2/\sum_i\sigma_i^2$) as a function of rank $k$. Find the smallest $k$ that captures 95% of the energy, and explain how this same curve, applied to a weight-update matrix instead of an image, is exactly how you'd choose a LoRA rank $r$ in practice.

---

## 📚 Further Reading
- Hu et al., ["LoRA: Low-Rank Adaptation of Large Language Models"](https://arxiv.org/abs/2106.09685)
- Strang, *Introduction to Linear Algebra*, Ch. 7 (Singular Value Decomposition)

---

*Next notebook: [Matrix Decompositions](08_matrix_decompositions.ipynb)*